## Introduction

In this notebook we explore supervised classification of astronomical spectra from the Sloan Digital Sky Survey (SDSS) using dimensionality reduction and machine learning techniques.

Astronomical spectra contain rich physical information about celestial objects, including continuum shape, emission and absorption lines, and ionization properties. However, spectra are intrinsically high-dimensional signals, making dimensionality reduction methods such as Principal Component Analysis (PCA) particularly useful for visualization and feature extraction.

Following the approach presented in the scikit-learn astronomy tutorial, each spectrum is first normalized using L2 normalization before PCA. This preprocessing step removes overall flux-amplitude differences caused by varying luminosities, cosmological distances, and observational conditions, allowing the analysis to focus primarily on spectral shape rather than absolute brightness.

It is important to note that PCA is therefore applied to normalized spectra, meaning that the extracted principal components describe dominant modes of variation in spectral morphology and continuum structure rather than raw flux variance.

The notebook compares multiple classification models, including Logistic Regression, Random Forests, and XGBoost, using cross-validated hyperparameter optimization within a leakage-safe scikit-learn pipeline. Model performance is evaluated on a fully independent test set using macro-averaged F1 score, accuracy, confusion matrices, sensitivity, and specificity metrics.

### Inspired by:

- https://github.com/csheneka/ML-for-Astro-tutorial/blob/main/spectral_classifier.ipynb
- https://ogrisel.github.io/scikit-learn.org/sklearn-tutorial/tutorial/astronomy/

## Imports

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import joblib
import scipy.stats as stats
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score


## Load Dataset

The Sloan Digital Sky Survey (SDSS) is a major astronomical survey that has been operating at Apache Point Observatory since 2000.
Up-to-date information can be found here: https://www.sdss.org/.

In this tutorial we use optical spectra obtained by the SDSS spectroscopic survey. The data contained in the `data_spectra/` folder is part of SDSS Data Release 16 (DR16).


In [ ]:
data_path = "../data/spectra/"

data = np.load(data_path + "data.npy")
labels = np.load(data_path + "labels.npy")
wavelengths = np.load(data_path + "wavelengths.npy")
class_names = ["AGN", "galaxy", "QSO"]

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)
print("Wavelength bins:", wavelengths.shape)

unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {class_names[u]}: {c} samples")


To gain an initial qualitative understanding of the dataset, we randomly visualize one spectrum from each astrophysical class.

This allows us to inspect the overall continuum shape and prominent spectral features characteristic of galaxies, AGN, and quasars.

In [ ]:
f, axs = plt.subplots(1, 3, figsize=(12,3))
rng = np.random.default_rng(12345)
for i,cls in enumerate(np.unique(labels)):
    idx = np.where(labels == cls)[0]
    n = rng.choice(idx)
    axs[i].plot(wavelengths, data[n], label=f"{class_names[labels[n]]}")
    axs[i].set_xlabel('wavelength(Å)')
    axs[i].set_title(f"{n} - {class_names[labels[n]]}")
    
axs[0].set_ylabel('flux (10-17 erg/s/cm2/Å)')
plt.tight_layout()
plt.show()


To better characterize the dataset, we compute the mean spectrum and standard deviation for each astrophysical class. 

This provides a compact representation of the typical spectral behavior and variability across wavelengths for galaxies, AGN, and quasars.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,12))
axs = axs.ravel()
colors = ["tab:blue", "tab:red", "tab:green"]
for i in (0, 1, 2):
    idx = np.where(labels == i)[0]
    mu = data[idx].mean(0)
    std = data[idx].std(0)
    l = axs[i].plot(wavelengths, mu, label=f"{class_names[i]}", color=colors[i])
    axs[i].fill_between(wavelengths, mu - std, mu + std, color=colors[i], alpha=0.3)
    axs[i].set_xlabel('wavelength (Å)')
    axs[i].set_ylim(-40, 80)
    axs[i].set_title(f"{class_names[i]}")
    
axs[0].set_ylabel('flux (10-17 erg/s/cm2/Å)')
axs[2].set_ylabel('flux (10-17 erg/s/cm2/Å)')
axs[-1].plot(wavelengths, data.mean(0), label=f"mean", color="black")
axs[-1].fill_between(wavelengths, data.mean(0) - data.std(0), data.mean(0) + data.std(0), color="grey", alpha=0.3)
axs[-1].set_xlabel('wavelength (Å)')
axs[-1].set_ylim(-40, 80)
axs[-1].set_title("mean")
plt.tight_layout()
plt.show()


We visualize the class distribution of the dataset to inspect the relative number of samples belonging to each astrophysical category.

In [ ]:
# Maps the integer labels to their actual string names for the plot
mapped_labels = [class_names[i] for i in labels]
sns.countplot(x=mapped_labels, order=class_names)
plt.title("Class Distribution")
plt.show()

## Data-driven Classification Pipeline

The dataset is divided into a training set and an independent test set using stratified sampling in order to preserve the original class distribution. 

Model selection and hyperparameter optimization are performed exclusively on the training data through cross-validation, while the test set remains completely unseen until the final evaluation stage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data,
    labels.astype("int32"),
    test_size=0.1,
    stratify=labels,
    random_state=42,
    shuffle=True
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

Because the spectra correspond to objects spanning a wide range of intrinsic luminosities and cosmological distances, their observed flux amplitudes can vary substantially. 

To reduce the impact of these global intensity differences and emphasize spectral shape, each spectrum is first normalized using L2 normalization before applying Principal Component Analysis (PCA).



In [ ]:
pipeline_pca = Pipeline([
    ("normalize", Normalizer(norm="l2")),
    ("pca", PCA(n_components=5, random_state=42))
])

X_proj = pipeline_pca.fit_transform(X_train)


We project the normalized spectra onto the first two principal components obtained from PCA and visualize the resulting 2D embedding. 

Each point represents one spectrum, colored by its true class, allowing us to assess how well different astrophysical populations separate in the reduced feature space.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

scatter = ax.scatter(
    X_proj[:, 0],
    X_proj[:, 1],
    c=y_train,
    s=4,
    cmap="viridis",
    alpha=0.8
)

cbar = fig.colorbar(scatter, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(class_names)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA Projection of SDSS Spectra")
plt.tight_layout()
plt.show()

Since PCA is applied to L2-normalized spectra, the resulting components describe variations in *spectral shape* rather than absolute flux levels.

The first principal component primarily captures large-scale continuum variations across wavelength, which are closely related to the overall spectral slope or “color” of the object.

The second component encodes more localized structure, including variations in emission and absorption features as well as contributions from the 4000 Å break.

Together, these eigenspectra provide a compact, interpretable basis for understanding dominant modes of spectral variation. However, PCA is a linear method and is not optimized for class separation, particularly when differences are driven by narrow spectral features.

> C.W. Yip et al., *Spectral Classification of Quasars in the Sloan Digital Sky Survey: Eigenspectra, Redshift, and Luminosity Effects*, Astronomical Journal 128:6 (2004).

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,4))
axs[1].plot(wavelengths, pipeline_pca.named_steps["pca"].mean_, label=f"Mean", color="tab:blue")
colors = ["tab:orange", "tab:red"]
for i in range(2):
    l = axs[1].plot(wavelengths, pipeline_pca.named_steps["pca"].components_[i], 
                    label=f"Component {(i + 1)}", color=colors[i])
    
axs[1].set_xlabel('Wavelength (Å)')
axs[1].set_ylabel('Scaled flux')
axs[1].set_title('Mean Spectrum and Eigen-spectra')
axs[1].legend()
axs[0].bar(np.arange(1, 6), pipeline_pca.named_steps["pca"].explained_variance_ratio_)

axs[0].set_xlabel("Principal Component")
axs[0].set_ylabel("Explained Variance Ratio")
plt.show()


## Classification Pipeline and Model Selection

We construct a flexible scikit-learn pipeline that combines optional preprocessing, dimensionality reduction via PCA, and a downstream classifier. We then define a unified hyperparameter search space to compare Logistic Regression, Random Forest, and XGBoost models under identical preprocessing conditions using randomized cross-validation search.

In [ ]:
# Base pipeline structure
pipeline = Pipeline([
    ("preprocess", "passthrough"),
    ("pca", "passthrough"),
    ("classifier", LogisticRegression())
])

# Define the search space for the models

models = {

    # Logistic Regression
    "logit": (
        LogisticRegression(
            max_iter=10000,
            solver="lbfgs", 
        ),
        {
            "preprocess": [
                           Normalizer(norm="l2")
                          ],
            "pca": [
                "passthrough",
                PCA(n_components=0.68, random_state=42),
                PCA(n_components=0.95, random_state=42),
            ],
            "classifier__C": stats.loguniform(1e-1, 1e4),
            "classifier__class_weight": [None, "balanced"]
        }
    ),

    # Random Forest
    "rf": (
        RandomForestClassifier(
            random_state=42,
            n_jobs=1
        ),
        {
            "preprocess": [
                            Normalizer(norm="l2")
                          ],

            "pca": [
                "passthrough",
                PCA(n_components=0.68, random_state=42),
                PCA(n_components=0.95, random_state=42),
            ],

            "classifier__n_estimators": stats.randint(100, 400),
            "classifier__max_depth": [5, 10, 20, None],
            "classifier__min_samples_leaf": [1, 2, 5],
            "classifier__max_features": ["sqrt", "log2"],
            "classifier__class_weight": [None, "balanced"]
        }
    ),

    # XGBoost
    "xgb": (
        xgb.XGBClassifier(
            eval_metric="mlogloss",      # Use multiclass logloss
            objective="multi:softprob",  # Explicitly state multiclass objective
            random_state=42,
            n_jobs=1
        ),
        {
            "preprocess": [
                            Normalizer(norm="l2")
                          ],
            "pca": [
                "passthrough",
                PCA(n_components=0.68, random_state=42),
                PCA(n_components=0.95, random_state=42),
            ],
            "classifier__n_estimators": stats.randint(100, 500),
            "classifier__learning_rate": stats.loguniform(1e-3, 1e-1),
            "classifier__max_depth": [3, 4, 5, 6],
            "classifier__subsample": [0.6, 0.8, 1.0],
            "classifier__colsample_bytree": [0.5, 0.8, 1.0],
            "classifier__min_child_weight": [1, 3, 5]
        }
    )
}

## Cross-Validation Strategy

We use stratified k-fold cross-validation to evaluate models in a robust and balanced way, ensuring that each fold preserves the original class distribution. 

This helps provide a more reliable estimate of model performance across the different astrophysical classes.


In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


# Hyperparameter Optimization

We perform model selection by iterating over different classifier families within a shared pipeline and optimizing their hyperparameters using randomized cross-validation. 

For each model, we train using stratified k-fold CV, evaluate performance with macro-averaged F1 score, and finally assess generalization on the held-out test set. 

The best-performing configuration is tracked based on cross-validation performance.

In [ ]:
results = []
trained_models = {}
best_overall_model = None
best_overall_score = -np.inf
best_model_name = None

for name, (model, params) in models.items():

    print(f"\n{'='*50}")
    print(f"Training {name}")
    print(f"{'='*50}")

    # configure pipeline
    pipeline.set_params(classifier=model)

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=10,                 # increased from 2
        cv=cv,
        scoring="f1_macro",
        random_state=42,
        verbose=1,
        n_jobs=-1,                 # use all CPU cores
        return_train_score=True
    )

    start = time.time()

    search.fit(X_train, y_train)

    elapsed = time.time() - start

    # store search object
    trained_models[name] = search

    # best estimator from CV
    best_model = search.best_estimator_

    # test predictions
    pred = best_model.predict(X_test)

    # metrics
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="macro")

    print("\nBest Parameters:")
    print(search.best_params_)

    print("\nBest CV Score:")
    print(f"{search.best_score_:.4f}")

    print("\nTest Metrics:")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1 Macro : {f1:.4f}")
    # save summary
    results.append({
        "model": name,
        "accuracy": acc,
        "f1_macro": f1,
        "best_cv_score": search.best_score_,
        "fit_time_sec": elapsed,
        "best_params": search.best_params_
    })

    # track best overall model
    if search.best_score_ > best_overall_score:
        best_overall_score = search.best_score_
        best_overall_model = best_model
        best_model_name = name

We collect the results from all evaluated models into a summary table and sort them by cross-validation performance. 

This allows a direct comparison of classifier performance under consistent evaluation metrics and preprocessing conditions.

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="best_cv_score",
    ascending=False
)

print("\nFinal Results:")
print(results_df)

## Report

We report the best-performing model selected based on cross-validation macro F1 score. 

This includes the model name, its validation performance, and the full fitted pipeline configuration.

In [ ]:
print("\n" + "="*60)
print("BEST OVERALL MODEL")
print("="*60)

print(f"Model: {best_model_name}")
print(f"CV F1 Macro: {best_overall_score:.4f}")

print("\nPipeline:")
print(best_overall_model)

## Save Model

We save the best-performing trained pipeline to disk using `joblib` for later reuse. 

This includes the full preprocessing, dimensionality reduction, and classification steps, allowing the model to be directly reloaded for inference without retraining.

In [ ]:
joblib.dump(best_overall_model, "best_spectrum_classifier_cpu.pkl")

## Define the Evaluation Metrics

We define evaluation utilities to assess classifier performance in detail. 

In addition to standard classification metrics, we compute per-class sensitivity and specificity from the confusion matrix, and visualize normalized confusion matrices to better understand class-wise prediction behavior.

In [ ]:
def get_multiclass_metrics(y_actual, y_pred, classes):
    cm = confusion_matrix(y_actual, y_pred)
    
    for i, cls in enumerate(classes):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - TP - FP - FN

        # Safe division to prevent ZeroDivisionError
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

        print(f"{cls:10s} Sensitivity: {sensitivity:.4f} | Specificity: {specificity:.4f}")

def report(y_actual, y_pred, classes, title):
    print(classification_report(y_actual, y_pred, target_names=classes))
    print("\nPer-class metrics:")
    get_multiclass_metrics(y_actual, y_pred, classes)

    cm = confusion_matrix(y_actual, y_pred, normalize="true")

    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="viridis", 
                xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

## Final Model Evaluation

We evaluate the final selected model on the held-out test set.

The trained pipeline is loaded from disk and used to generate predictions, which are then assessed using accuracy and macro-averaged F1 score to quantify generalization performance.

In [ ]:
# We load the best refitted model
final_model = joblib.load("best_spectrum_classifier_cpu.pkl")
test_predictions = final_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_predictions)
test_f1 = f1_score(y_test, test_predictions, average="macro")

print(f"Final Test Accuracy: {test_accuracy:.4f}")
print(f"Final Test F1: {test_f1:.4f}")

We perform a detailed evaluation of the final model on the test set, including per-class metrics and a normalized confusion matrix to visualize classification performance across all astrophysical categories.

In [ ]:
report(y_test, test_predictions, class_names, title="Final Test Confusion Matrix")